In [12]:
pip install datasets evaluate rouge-score sacrebleu


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=525ddf896437268c910be0031cbb9f6767374fd7df742db7e3ff8fc916f8f30b
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

2025-06-24 14:24:08.303502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750775048.492863      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750775048.550202      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
raw_train = load_dataset("euclaise/writingprompts", split="train")

raw_train = raw_train.select(range(10000))

split_dataset = raw_train.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

README.md:   0%|          | 0.00/837 [00:00<?, ?B/s]

(…)-00000-of-00002-105e07cb0d199464.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

(…)-00001-of-00002-4fdb982c11056472.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

(…)-00000-of-00001-16503b0c26ed00c6.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

(…)-00000-of-00001-137b93e1e979d138.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/272600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15138 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15620 [00:00<?, ? examples/s]

In [6]:
# Check the size of the datasets
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")


Training dataset size: 9000
Validation dataset size: 1000


In [7]:
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [29]:
import evaluate
import math
import torch

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute metrics
    result = {
        "bleu": bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"],
        "rougeL": rouge.compute(predictions=decoded_preds, references=decoded_labels)["rougeL"]
    }

    # Try to compute perplexity if logits are available
    try:
        logits = torch.tensor(preds)
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = torch.tensor(labels[..., 1:]).contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        result["perplexity"] = round(math.exp(loss.item()), 4)
    except:
        result["perplexity"] = float("nan")

    return {k: round(v, 4) for k, v in result.items()}


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


/tmp/ipykernel_35/4159831530.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
trainer.train()
trainer.save_model("./content_creation")

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.251800,3.140676
2,3.117700,3.124797
3,3.059300,3.120916


In [18]:
tokenizer.save_pretrained("./content_creation")


('./content_creation/tokenizer_config.json',
 './content_creation/special_tokens_map.json',
 './content_creation/vocab.json',
 './content_creation/merges.txt',
 './content_creation/added_tokens.json',
 './content_creation/tokenizer.json')

In [20]:
!zip -r content_creation.zip ./content_creation


  adding: content_creation/ (stored 0%)
  adding: content_creation/special_tokens_map.json (deflated 60%)
  adding: content_creation/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: content_creation/generation_config.json (deflated 24%)
  adding: content_creation/vocab.json (deflated 59%)
  adding: content_creation/tokenizer.json (deflated 82%)
  adding: content_creation/config.json (deflated 52%)
  adding: content_creation/tokenizer_config.json (deflated 54%)
  adding: content_creation/training_args.bin (deflated 52%)
  adding: content_creation/merges.txt (deflated 53%)


In [14]:
trainer.train(resume_from_checkpoint='/home/deepan/AnanditaUma/contentcreation/results/checkpoint-33750/')

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Epoch,Training Loss,Validation Loss
4,1.509300,3.030557
5,1.500500,3.020418
6,1.510700,3.016378


/home/deepan/AnanditaUma/translation/translation/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/deepan/AnanditaUma/translation/translation/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/deepan/AnanditaUma/translation/translation/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=67500, training_loss=0.7600976675528067, metrics={'train_runtime': 11526.1109, 'train_samples_per_second': 46.85, 'train_steps_per_second': 5.856, 'total_flos': 2.8219539456e+17, 'train_loss': 0.7600976675528067, 'epoch': 6.0})

In [ ]:
# Initialize with proper device handling
tokenizer = AutoTokenizer.from_pretrained("./gpt2-writingprompts2")
model = AutoModelForCausalLM.from_pretrained("./gpt2-writingprompts2")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A robot wakes up on a deserted planet.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=512,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: A robot wakes up on a deserted planet , as it discovers the human race is on a colony ship
 It had been a long day for the ship. It was too late to go home. 
 
 A low hum was heard, a faint, rasping sound. The ship was empty. 
 
 The ship was empty. 
 
 No. 
 
 *You're awake. * 
 
 I could not move. 
 
 It was not my fault. 
 
 I woke up in a lab room with a small black lab in my arms. There were two metal chairs, one with two monitors and one with three monitors. A small computer, a single console, a single display, and a computer monitor. 
 
 *Where the hell is all this? * 
 
 I moved my head to look around the room, it was filled with the kind of people I would never meet. I looked around and it was all the same to me. 
 
 The chair was open. I felt a small hand, I looked up and saw two other men. One was dressed in what looked like an army uniform, the other one with a black hoodie. 
 
 `` What's this?'' The other asked. 
 
 `` This is the ship, and this is the ship. The sh

In [21]:
model_name = "/kaggle/working/content_creation"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

In [22]:
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [24]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [26]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


/tmp/ipykernel_35/244447918.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [28]:
trainer.train()
trainer.save_model("./content_creation2")
tokenizer.save_pretrained("./content_creation2")
!zip -r content_creation2.zip ./content_creation2


Epoch,Training Loss,Validation Loss
1,3.065800,3.125058


  adding: content_creation2/ (stored 0%)
  adding: content_creation2/special_tokens_map.json (deflated 74%)
  adding: content_creation2/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: content_creation2/generation_config.json (deflated 24%)
  adding: content_creation2/vocab.json (deflated 59%)
  adding: content_creation2/tokenizer.json (deflated 82%)
  adding: content_creation2/config.json (deflated 52%)
  adding: content_creation2/tokenizer_config.json (deflated 57%)
  adding: content_creation2/training_args.bin (deflated 52%)
  adding: content_creation2/merges.txt (deflated 53%)


In [ ]:
# Replace with your own tokenized test set
metrics = trainer.evaluate(eval_dataset=tokenized_test)
print(metrics)


In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("./content_creation")
tokenizer = AutoTokenizer.from_pretrained("./content_creation")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    
import evaluate
import math
import torch

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

BLEU: 0.42
ROUGE-L: 0.09
Perplexity: 24.1553


In [38]:
!pip install evaluate
!pip install bert_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:01m0:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.6 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 44.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing inst

In [39]:
import evaluate

bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.7933246456384658


In [52]:
# Initialize with proper device handling
tokenizer = AutoTokenizer.from_pretrained("./content_creation")
model = AutoModelForCausalLM.from_pretrained("./content_creation")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A man wakes up on a deserted island.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=128,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: A man wakes up on a deserted island.
 This guy woke up like everyone else on that island. He was in the back of his ship, and all he could think was how he could make it to the other end of the ship. He did n't want to be there; he needed to get home. He needed to find a way to leave. 
 
 
 There are some places where you can sleep on empty. The back door is open, but the only one he has is on the other side of it. There is no way to get home, so he made his way to the


In [53]:

model = AutoModelForCausalLM.from_pretrained("./content_creation2")
tokenizer = AutoTokenizer.from_pretrained("./content_creation2")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

BLEU: 0.42
ROUGE-L: 0.09
Perplexity: 24.384


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.7931430215239524


In [2]:
pip install datasets evaluate rouge-score sacrebleu


Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch
raw_train = load_dataset("euclaise/writingprompts", split="train")

raw_train = raw_train.select(range(10000))

split_dataset = raw_train.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
# Check the size of the datasets
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
model_name = "distilgpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


2025-06-25 18:21:45.985268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750875706.187959      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750875706.245852      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


README.md:   0%|          | 0.00/837 [00:00<?, ?B/s]

(…)-00000-of-00002-105e07cb0d199464.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

(…)-00001-of-00002-4fdb982c11056472.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

(…)-00000-of-00001-16503b0c26ed00c6.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_35/528849018.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [5]:
trainer.train()
trainer.save_model("./content_creation-small")
tokenizer.save_pretrained("./content_creation-small")
!zip -r content_creation-small.zip ./content_creation-small

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.508800,3.384867
2,3.384500,3.359564
3,3.335200,3.354442


  adding: content_creation-small/ (stored 0%)
  adding: content_creation-small/tokenizer_config.json (deflated 54%)
  adding: content_creation-small/training_args.bin (deflated 52%)
  adding: content_creation-small/generation_config.json (deflated 24%)
  adding: content_creation-small/vocab.json (deflated 59%)
  adding: content_creation-small/config.json (deflated 52%)
  adding: content_creation-small/special_tokens_map.json (deflated 60%)
  adding: content_creation-small/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 82%)
  adding: content_creation-small/merges.txt (deflated 53%)
  adding: content_creation-small/model.safetensors (deflated 7%)


In [10]:
import evaluate
import math
import torch

model = AutoModelForCausalLM.from_pretrained("./content_creation-small")
tokenizer = AutoTokenizer.from_pretrained("./content_creation-small")
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
test_prompts = val_dataset["prompt"]
true_stories = val_dataset["story"]
model.eval()
model.to("cuda")  # or "cpu" if no GPU

generated_stories = []
batch_size = 4
max_length = 256

for i in range(0, len(test_prompts), batch_size):
    batch_prompts = test_prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,  # greedy
            num_beams=4       # or adjust
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generated_stories.extend(decoded)
    

# Load metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_score = bleu.compute(predictions=generated_stories, references=[[ref] for ref in true_stories])["score"]
rouge_score = rouge.compute(predictions=generated_stories, references=true_stories)["rougeL"]

# Perplexity (approximate using loss on ground truth)
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    losses = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return round(math.exp(mean_loss), 4)

perplexity_score = compute_perplexity(model, tokenizer, true_stories[:100])  
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_score:.2f}")
print(f"Perplexity: {perplexity_score}")
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

BLEU: 0.40
ROUGE-L: 0.09
Perplexity: 30.2741


ImportError: To be able to use evaluate-metric/bertscore, you need to install the following dependencies['bert_score'] using 'pip install bert_score' for instance'

In [14]:
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=generated_stories, references=true_stories, lang="en")
print("BERTScore F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.7894456019997597


In [13]:
pip install bert_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing ins

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch
raw_train = load_dataset("euclaise/writingprompts", split="train")

raw_train = raw_train.select(range(10000))

split_dataset = raw_train.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
# Check the size of the datasets
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
model_name = "/kaggle/working/content_creation-small"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
def tokenize_function(examples):
    texts = [p + " " + s for p, s in zip(examples["prompt"], examples["story"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=tokenizer.model_max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["prompt", "story"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",         
    save_strategy="no",            
    logging_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)


trainer1 = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


Training dataset size: 9000
Validation dataset size: 1000


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_35/2224379102.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer1 = Trainer(


In [7]:
trainer1.train()
trainer1.save_model("./content_creation-small2")
tokenizer.save_pretrained("./content_creation-small2")
!zip -r content_creation-small2.zip ./content_creation-small2

Epoch,Training Loss,Validation Loss
1,3.301700,3.356187


  adding: content_creation-small2/ (stored 0%)
  adding: content_creation-small2/tokenizer_config.json (deflated 57%)
  adding: content_creation-small2/training_args.bin (deflated 52%)
  adding: content_creation-small2/generation_config.json (deflated 24%)
  adding: content_creation-small2/vocab.json (deflated 59%)
  adding: content_creation-small2/config.json (deflated 52%)
  adding: content_creation-small2/special_tokens_map.json (deflated 74%)
  adding: content_creation-small2/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 82%)
  adding: content_creation-small2/merges.txt (deflated 53%)
  adding: content_creation-small2/model.safetensors (deflated 7%)


In [20]:
# Initialize with proper device handling
tokenizer = AutoTokenizer.from_pretrained("./content_creation-small2")
model = AutoModelForCausalLM.from_pretrained("./content_creation-small2")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Model first

# Tokenize WITH device transfer
input_ids = tokenizer(
    "Prompt: A robot wakes up on a deserted island.",
    return_tensors="pt"
).to(device).input_ids  

# Generate
model.eval()
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=128,
        do_sample=True,
        top_p=0.9,
        temperature=0.9
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: A robot wakes up on a deserted island. But why? Why? Why do n't they have to resort to violence? The robots are humans. They have been trained to be deadly, they know that when they die they will not be given life. And it will not be easy. There are many people in this world, but they have to survive. And so they must survive. They must survive. 
 
 I do n't know why, but I have n't been able to do it. I've found many ways to survive. I've found a place in a forest, a land mine,


In [19]:
|

SyntaxError: invalid syntax (525519296.py, line 1)